[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/09_attention_advanced.ipynb)

# 09. Advanced attention

attention score/normalization과 연결 패턴을 바꾸는 변형을 작은 tensor에서 비교한다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


In [ ]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. Local / sliding-window mask

각 token이 가까운 이웃만 본다.


In [ ]:
T = 6
window = 1
idx = torch.arange(T, device=device)
mask = (idx[:, None] - idx[None, :]).abs() > window

q = torch.randn(1, 1, T, 4, device=device)
k = torch.randn_like(q)
v = torch.randn_like(q)

out = F.scaled_dot_product_attention(q, k, v, attn_mask=~mask)
print("mask:\n", (~mask).int())
print("out shape:", out.shape)


In [ ]:
_ = profile_call("local attention", lambda: F.scaled_dot_product_attention(q, k, v, attn_mask=~mask))


## 2. Top-k sparse attention

score에서 상위 k개만 남긴다.


In [ ]:
scores = (q @ k.transpose(-2, -1)) / math.sqrt(q.size(-1))
topv, topi = scores.topk(k=2, dim=-1)

sparse_scores = torch.full_like(scores, float("-inf"))
sparse_scores.scatter_(-1, topi, topv)

weights = sparse_scores.softmax(-1)
out_sparse = weights @ v

print("selected indices:\n", topi)


In [ ]:
def sparse_attention_once():
    sparse_ = torch.full_like(scores, float("-inf"))
    sparse_ = sparse_.scatter(-1, topi, topv)
    return sparse_.softmax(-1) @ v

_ = profile_call("top-k sparse attention", sparse_attention_once)


## 3. Sigmoid attention

softmax 대신 각 score를 독립 sigmoid gate로 사용한다.


In [ ]:
scores = (q @ k.transpose(-2, -1)) / math.sqrt(q.size(-1))
sig_weights = torch.sigmoid(scores)
sig_out = sig_weights @ v

print("sigmoid weight row sums:", sig_weights.sum(-1))


In [ ]:
_ = profile_call("sigmoid attention", lambda: torch.sigmoid(scores) @ v)


## 4. Gated attention

attention 출력에 별도 gate를 곱한다.


In [ ]:
attn = F.scaled_dot_product_attention(q, k, v)
gate = torch.sigmoid(torch.randn(1, 1, T, 1, device=device))
gated = gate * attn

print("gate:", gate.flatten())
print("gated output norm:", gated.norm().item())


In [ ]:
_ = profile_call("gated attention", lambda: gate * F.scaled_dot_product_attention(q, k, v))


## 5. Delta-style recurrent update

새 key/value가 상태를 rank-1 형태로 갱신하는 핵심 아이디어만 본다.


In [ ]:
D = 4
state = torch.zeros(D, D, device=device)
key = F.normalize(torch.tensor([1., 2., 1., 0.], device=device), dim=0)
value = torch.tensor([0.5, -1., 2., 1.], device=device)
beta = torch.tensor(0.4, device=device)

prediction = state @ key
state = state + beta * torch.outer(value - prediction, key)

print("prediction before update:", prediction)
print("state after delta update:\n", state)


In [ ]:
_ = profile_call("delta state update", lambda s, k_, v_: s + beta * torch.outer(v_ - s @ k_, k_), state, key, value)


## References and provenance

**[9.1] Sliding-window attention**
- 출처: Longformer/Mistral lineage
- 이 노트북에서 가져온 부분: local connectivity

**[9.2] Sparse attention**
- 출처: DeepSeek sparse-attention research lineage
- 이 노트북에서 가져온 부분: selected score computation

**[9.3] Sigmoid/gated attention**
- 출처: recent DiT/LLM reports including Krea-family designs
- 이 노트북에서 가져온 부분: replace/augment softmax with gates

**[9.4] Delta/KDA-style recurrence**
- 출처: DeltaNet / Kimi linear-attention research lineage
- 이 노트북에서 가져온 부분: state update instead of full attention matrix
